# 2.4 · Validación walk-forward y comparación final

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Implementar **walk-forward** sobre el caudal mensual del Genil.
2. Calcular **NSE, KGE, error en pico** + RMSE, MAE.
3. Comparar SARIMAX vs ETS vs Prophet con la misma metodología.

## Mini-intro (10 min)

**Split temporal único** (lo que hicimos en 01/02/03):

```
[============ train ============][===== test =====]
```

Limitación: una sola estimación del error. ¿Y si el test cae en un periodo atípico?

**Walk-forward / expanding window:**

```
[===== train ======][test]
[======= train =======][test]
[========= train =========][test]
...
```

El modelo se reajusta cada paso; se predice un horizonte fijo (1 o varios pasos) y se acumulan errores. Da una distribución, no un punto.

## Métricas hidrológicas

- **NSE** (Nash-Sutcliffe Efficiency): $1 - \frac{\sum (o - p)^2}{\sum (o - \bar{o})^2}$.
  - 1 = perfecto. 0 = igual que la media. < 0 = peor que predecir la media.
- **KGE** (Kling-Gupta Efficiency): combina correlación, ratio de varianzas y sesgo.
  $$\mathrm{KGE} = 1 - \sqrt{(r-1)^2 + (\alpha-1)^2 + (\beta-1)^2}$$
  donde $\alpha = \sigma_p/\sigma_o$, $\beta = \mu_p/\mu_o$.
- **Error en pico**: $\max(p) - \max(o)$ dentro del periodo de evaluación. Crítico para crecidas.
- **Diebold-Mariano** (mención): test estadístico para comparar dos modelos de forecast.

In [ ]:
import logging

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})

## 1 · Métricas (implementación)

In [ ]:
def nse(obs, sim):
    obs, sim = np.asarray(obs), np.asarray(sim)
    return 1 - np.sum((obs - sim) ** 2) / np.sum((obs - obs.mean()) ** 2)


def kge(obs, sim):
    obs, sim = np.asarray(obs), np.asarray(sim)
    r = np.corrcoef(obs, sim)[0, 1]
    alpha = sim.std() / obs.std()
    beta = sim.mean() / obs.mean()
    return 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


def error_pico(obs, sim):
    return float(np.max(sim) - np.max(obs))


def rmse(obs, sim):
    return float(np.sqrt(np.mean((np.asarray(obs) - np.asarray(sim)) ** 2)))


def mae(obs, sim):
    return float(np.mean(np.abs(np.asarray(obs) - np.asarray(sim))))


# Sanity check con un caso perfecto
y = np.array([1.0, 2.0, 3.0, 4.0])
print("NSE perfecto:", nse(y, y), "  KGE perfecto:", kge(y, y))

## 2 · Datos y rolling forecast (horizonte 1 mes)

Reajustamos cada modelo en cada paso y predecimos 1 mes. Mantenemos `2015-01-01` → final como periodo de evaluación.

In [ ]:
caudal_d = ud.cargar_caudal_genil(source="CEDEX")
lluvia_d = ud.cargar_lluvia_genil_diaria(fecha_inicio="1995-01-01", fecha_fin="2020-12-31")
caudal_m = caudal_d.resample("MS").mean().loc["1995":"2020"].interpolate("linear", limit=2).dropna()
lluvia_m = lluvia_d.resample("MS").sum().loc["1995":"2020"]
df = pd.DataFrame({"y": caudal_m, "lluvia": lluvia_m}).dropna()

inicio_eval = pd.Timestamp("2015-01-01")
fechas_eval = df.loc[inicio_eval:].index
print(f"Pasos de walk-forward: {len(fechas_eval)}")

In [ ]:
preds = {modelo: [] for modelo in ["SARIMAX", "ETS", "Prophet"]}
obs = []

for fecha in fechas_eval:
    historico = df.loc[:fecha].iloc[:-1]  # todo hasta el mes anterior
    objetivo = df.loc[fecha]
    obs.append(objetivo["y"])

    # --- SARIMAX(1,1,1)(1,0,1,12) con lluvia retardada ---
    exog_h = historico["lluvia"].shift(1).fillna(method="bfill").values.reshape(-1, 1)
    try:
        m_sx = SARIMAX(
            historico["y"],
            exog=exog_h,
            order=(1, 1, 1),
            seasonal_order=(1, 0, 1, 12),
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False, maxiter=50)
        exog_fut = np.array([[historico["lluvia"].iloc[-1]]])
        preds["SARIMAX"].append(float(m_sx.forecast(steps=1, exog=exog_fut).iloc[0]))
    except Exception:
        preds["SARIMAX"].append(np.nan)

    # --- ETS Holt-Winters ---
    try:
        m_ets = ExponentialSmoothing(
            historico["y"],
            trend="add",
            seasonal="add",
            seasonal_periods=12,
            initialization_method="estimated",
        ).fit()
        preds["ETS"].append(float(m_ets.forecast(1).iloc[0]))
    except Exception:
        preds["ETS"].append(np.nan)

    # --- Prophet ---
    try:
        prh_df = historico.rename_axis("ds").reset_index()
        prh_df = prh_df.rename(columns={"y": "y"})
        m_p = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        m_p.add_regressor("lluvia")
        m_p.fit(prh_df)
        fut = pd.DataFrame({"ds": [fecha], "lluvia": [historico["lluvia"].iloc[-1]]})
        preds["Prophet"].append(float(m_p.predict(fut)["yhat"].iloc[0]))
    except Exception:
        preds["Prophet"].append(np.nan)

obs = np.array(obs)
for k in preds:
    preds[k] = np.array(preds[k])
print("Walk-forward completado.")

## 3 · Plot y métricas

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(fechas_eval, obs, color="black", lw=1.5, label="observado")
colores = {"SARIMAX": "#c2410c", "ETS": "#16a34a", "Prophet": "#7c3aed"}
for k, p in preds.items():
    ax.plot(fechas_eval, p, color=colores[k], lw=1, ls="--", label=k)
ax.set_ylabel("Q (m³/s)")
ax.legend()
ax.set_title("Walk-forward 2015-2020 — horizonte 1 mes")
plt.tight_layout()

In [ ]:
filas = []
for nombre, p in preds.items():
    mask = ~np.isnan(p)
    filas.append(
        {
            "modelo": nombre,
            "NSE": round(nse(obs[mask], p[mask]), 3),
            "KGE": round(kge(obs[mask], p[mask]), 3),
            "RMSE": round(rmse(obs[mask], p[mask]), 3),
            "MAE": round(mae(obs[mask], p[mask]), 3),
            "Error pico": round(error_pico(obs[mask], p[mask]), 3),
        }
    )
pd.DataFrame(filas).set_index("modelo")

## 4 · Lectura

- **NSE ≥ 0.5** suele considerarse aceptable para predicción mensual de caudal en cuencas reguladas.
- **KGE > NSE** habitualmente porque penaliza menos el sesgo cuando la variabilidad sí está bien capturada.
- **Error en pico**: incluso con buen NSE, el modelo puede infraestimar sistemáticamente los picos (negativo).

Cuál es mejor depende de para qué: gestión de embalse (foco en medias) ≠ alerta de avenida (foco en picos).

## 5 · Ejercicios

1. **Horizonte 3.** Repite el walk-forward con horizonte de 3 meses. ¿Qué modelo aguanta mejor?
2. **Bootstrap del NSE.** Calcula el IC del 95% del NSE por bootstrap de los pares (obs, sim).
3. **Sólo crecidas.** Subselecciona los meses con observado en el cuartil superior y recalcula métricas. ¿Cambia el ranking?
4. **Diebold-Mariano.** Implementa el test DM para comparar SARIMAX vs ETS. Conclusion ¿son estadísticamente distintos?
5. **Reto.** Añade un baseline persistencia (`yhat_t = y_{t-12}`) y compáralo con los tres modelos. En cuencas reguladas a menudo es muy difícil de batir.